# Celeb-DF v2 Deepfake Detection — EfficientNet-B0


## 1. Mount Google Drive and set persistent paths

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = "/content/drive/MyDrive/Deepfake_Project"
FRAME_ROOT = os.path.join(PROJECT_DIR, "celebdf_frames")
MANIFEST_DIR = os.path.join(PROJECT_DIR, "manifests")
CHECKPOINT_PATH = os.path.join(PROJECT_DIR, "last_checkpoint.pth")
BEST_MODEL_PATH = os.path.join(PROJECT_DIR, "best_model.pth")

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(FRAME_ROOT, exist_ok=True)
os.makedirs(MANIFEST_DIR, exist_ok=True)

print("Project folder:", PROJECT_DIR)
print("Frames folder:", FRAME_ROOT)


Mounted at /content/drive
Project folder: /content/drive/MyDrive/Deepfake_Project
Frames folder: /content/drive/MyDrive/Deepfake_Project/celebdf_frames


## 2. Install and download Celeb-DF v2

In [3]:
!pip -q install kagglehub opencv-python-headless scikit-learn matplotlib tqdm

import kagglehub

dataset_path = kagglehub.dataset_download("reubensuju/celeb-df-v2")
print("Dataset downloaded to:", dataset_path)

print("\nTop-level contents:")
for item in os.listdir(dataset_path):
    full_path = os.path.join(dataset_path, item)
    print(item, "—", "folder" if os.path.isdir(full_path) else "file")


Using Colab cache for faster access to the 'celeb-df-v2' dataset.
Dataset downloaded to: /kaggle/input/celeb-df-v2

Top-level contents:
YouTube-real — folder
Celeb-synthesis — folder
Celeb-real — folder
List_of_testing_videos.txt — file


## 3. Build the video dataframe

In [4]:
import pandas as pd

real_folders = ["YouTube-real", "Celeb-real"]
fake_folders = ["Celeb-synthesis"]

records = []

for folder in real_folders:
    folder_path = os.path.join(dataset_path, folder)
    if not os.path.isdir(folder_path):
        raise FileNotFoundError(f"Required folder not found: {folder_path}")

    for filename in os.listdir(folder_path):
        if filename.lower().endswith((".mp4", ".avi", ".mov")):
            records.append({
                "path": os.path.join(folder_path, filename),
                "video": filename,
                "label": 0,
                "class": "REAL"
            })

for folder in fake_folders:
    folder_path = os.path.join(dataset_path, folder)
    if not os.path.isdir(folder_path):
        raise FileNotFoundError(f"Required folder not found: {folder_path}")

    for filename in os.listdir(folder_path):
        if filename.lower().endswith((".mp4", ".avi", ".mov")):
            records.append({
                "path": os.path.join(folder_path, filename),
                "video": filename,
                "label": 1,
                "class": "FAKE"
            })

df = pd.DataFrame(records)

print("Total videos:", len(df))
print("\nClass counts:")
print(df["class"].value_counts())
display(df.head())


Total videos: 6529

Class counts:
class
FAKE    5639
REAL     890
Name: count, dtype: int64


,path,video,label,class
0,/kaggle/input/celeb-df-v2/YouTube-real/00238.mp4,00238.mp4,0,REAL
1,/kaggle/input/celeb-df-v2/YouTube-real/00152.mp4,00152.mp4,0,REAL
2,/kaggle/input/celeb-df-v2/YouTube-real/00269.mp4,00269.mp4,0,REAL
3,/kaggle/input/celeb-df-v2/YouTube-real/00209.mp4,00209.mp4,0,REAL
4,/kaggle/input/celeb-df-v2/YouTube-real/00297.mp4,00297.mp4,0,REAL


## 4. Create train, validation, and official test splits

In [5]:
from sklearn.model_selection import train_test_split

test_list_path = os.path.join(dataset_path, "List_of_testing_videos.txt")
if not os.path.exists(test_list_path):
    raise FileNotFoundError(f"Official test list not found: {test_list_path}")

with open(test_list_path, "r") as f:
    test_relpaths = {
        line.strip().split(maxsplit=1)[1].replace("\\", "/")
        for line in f
        if line.strip() and len(line.strip().split(maxsplit=1)) == 2
    }

df["relative_path"] = df["path"].apply(
    lambda p: os.path.relpath(p, dataset_path).replace("\\", "/")
)

df["split"] = "remaining"
df.loc[df["relative_path"].isin(test_relpaths), "split"] = "test"

matched = (df["split"] == "test").sum()
print("Official test videos matched:", matched)
print("Test-list entries:", len(test_relpaths))

if matched == 0:
    raise ValueError(
        "No official test-list entries matched the discovered videos. "
        "Check the dataset folder structure and test-list path format."
    )

remaining = df[df["split"] == "remaining"].copy()

train_df, val_df = train_test_split(
    remaining,
    test_size=0.15,
    random_state=42,
    stratify=remaining["label"]
)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = df[df["split"] == "test"].copy()

train_df["split"] = "train"
val_df["split"] = "val"

for name, part in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"\n{name}: {len(part)}")
    print(part["class"].value_counts())

train_csv = os.path.join(MANIFEST_DIR, "train_videos.csv")
val_csv = os.path.join(MANIFEST_DIR, "val_videos.csv")
test_csv = os.path.join(MANIFEST_DIR, "test_videos.csv")

train_df.to_csv(train_csv, index=False)
val_df.to_csv(val_csv, index=False)
test_df.to_csv(test_csv, index=False)

print("\nSplit manifests saved in:", MANIFEST_DIR)


Official test videos matched: 518
Test-list entries: 518

Train: 5109
class
FAKE    4504
REAL     605
Name: count, dtype: int64

Validation: 902
class
FAKE    795
REAL    107
Name: count, dtype: int64

Test: 518
class
FAKE    340
REAL    178
Name: count, dtype: int64

Split manifests saved in: /content/drive/MyDrive/Deepfake_Project/manifests


## 5. Extract frames (skip existing frames)

Five frames are sampled evenly from each video. Frames are stored in Google Drive. If a video's expected frame files already exist, extraction is skipped, which saves time after a runtime restart.

In [6]:
import cv2
import numpy as np
from tqdm.auto import tqdm

FRAMES_PER_VIDEO = 5
VIDEO_EXTENSIONS = (".mp4", ".avi", ".mov")

for split_name, split_df in [
    ("train", train_df),
    ("val", val_df),
    ("test", test_df)
]:
    for _, row in tqdm(
        split_df.iterrows(),
        total=len(split_df),
        desc=f"Extracting {split_name}"
    ):
        video_path = row["path"]
        label_name = row["class"]

        output_dir = os.path.join(FRAME_ROOT, split_name, label_name)
        os.makedirs(output_dir, exist_ok=True)

        video_id = os.path.splitext(os.path.basename(video_path))[0]
        expected_count = min(
            FRAMES_PER_VIDEO,
            max(0, int(cv2.VideoCapture(video_path).get(cv2.CAP_PROP_FRAME_COUNT)))
        )
        expected_paths = [
            os.path.join(output_dir, f"{video_id}_{i}.jpg")
            for i in range(expected_count)
        ]

        if expected_paths and all(os.path.exists(p) for p in expected_paths):
            continue

        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print("Could not open:", video_path)
            cap.release()
            continue

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames <= 0:
            cap.release()
            continue

        frame_indices = np.linspace(
            0, total_frames - 1,
            min(FRAMES_PER_VIDEO, total_frames),
            dtype=int
        )

        for i, frame_idx in enumerate(frame_indices):
            output_path = os.path.join(output_dir, f"{video_id}_{i}.jpg")
            if os.path.exists(output_path):
                continue

            cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
            success, frame = cap.read()
            if not success:
                continue

            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (224, 224))

            cv2.imwrite(
                output_path,
                cv2.cvtColor(frame, cv2.COLOR_RGB2BGR),
                [cv2.IMWRITE_JPEG_QUALITY, 90]
            )

        cap.release()

print("Frame extraction complete!")


Extracting train:   0%|          | 0/5109 [00:00<?, ?it/s]

Extracting val:   0%|          | 0/902 [00:00<?, ?it/s]

Extracting test:   0%|          | 0/518 [00:00<?, ?it/s]

Frame extraction complete!


## 6. Verify extracted frames

In [8]:
for split_name in ["train", "val", "test"]:
    for class_name in ["FAKE", "REAL"]:
        folder = os.path.join(FRAME_ROOT, split_name, class_name)
        count = (
            len([f for f in os.listdir(folder) if f.lower().endswith(".jpg")])
            if os.path.isdir(folder) else 0
        )
        print(f"{split_name:5} | {class_name:4} | {count} frames")


train | FAKE | 22520 frames
train | REAL | 3021 frames
val   | FAKE | 3975 frames
val   | REAL | 535 frames
test  | FAKE | 1700 frames
test  | REAL | 890 frames


## 7. Create datasets and data loaders

In [9]:
import torch
import torch.nn as nn
import copy
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_ds = datasets.ImageFolder(
    os.path.join(FRAME_ROOT, "train"),
    transform=train_transform
)
val_ds = datasets.ImageFolder(
    os.path.join(FRAME_ROOT, "val"),
    transform=eval_transform
)

print("Class mapping:", train_ds.class_to_idx)
print("Training frames:", len(train_ds))
print("Validation frames:", len(val_ds))

if len(train_ds) == 0 or len(val_ds) == 0:
    raise ValueError("Training or validation dataset is empty. Check frame extraction.")

# ImageFolder sorts alphabetically: FAKE=0, REAL=1
class_counts = np.bincount(train_ds.targets)
class_weights = len(train_ds) / (len(class_counts) * class_counts)
class_weights = torch.tensor(
    class_weights, dtype=torch.float32, device=device
)
print("Class counts [FAKE, REAL]:", class_counts)
print("Class weights:", class_weights.tolist())

# num_workers=0 is generally more reliable in Colab notebooks
train_loader = DataLoader(
    train_ds, batch_size=32, shuffle=True,
    num_workers=0, pin_memory=(device.type == "cuda")
)
val_loader = DataLoader(
    val_ds, batch_size=32, shuffle=False,
    num_workers=0, pin_memory=(device.type == "cuda")
)


Using device: cuda
Class mapping: {'FAKE': 0, 'REAL': 1}
Training frames: 25541
Validation frames: 4510
Class counts [FAKE, REAL]: [22520  3021]
Class weights: [0.5670737028121948, 4.227242469787598]


## 8. Train EfficientNet-B0 with automatic checkpoint resume


In [14]:

# Training settings
EPOCHS = 15
FREEZE_EPOCHS = 2
PATIENCE = 4

BEST_MODEL_PATH = "/content/drive/MyDrive/Deepfake_Project/best_model_finetune.pth"
CHECKPOINT_PATH = "/content/drive/MyDrive/Deepfake_Project/last_checkpoint_finetune.pth"

# Load EfficientNetB0
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

model.classifier[1] = nn.Linear(
    model.classifier[1].in_features, 2
)

# Freeze feature layers
for param in model.features.parameters():
    param.requires_grad = False

model = model.to(device)

# Define loss and optimizer
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(
    model.classifier.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

# Initialize training values
start_epoch = 0
best_f1 = 0.0
best_val_loss = float("inf")
early_stop_count = 0

# Load previous checkpoint if available
if os.path.exists(CHECKPOINT_PATH):

    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)

    model.load_state_dict(checkpoint["model_state_dict"])

    start_epoch = checkpoint["epoch"] + 1
    best_f1 = checkpoint["best_f1"]
    best_val_loss = checkpoint["best_val_loss"]
    early_stop_count = checkpoint["early_stop_count"]

    # Restore fine-tuning configuration
    if start_epoch >= FREEZE_EPOCHS:

        for param in model.features[-3:].parameters():
            param.requires_grad = True

        optimizer = torch.optim.AdamW(
            [
                {"params": model.features[-3:].parameters(), "lr": 1e-5},
                {"params": model.classifier.parameters(), "lr": 1e-4}
            ],
            weight_decay=1e-4
        )

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=2
        )

        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    print("Checkpoint loaded!")
    print("Resuming from epoch:", start_epoch + 1)

else:
    print("Starting new training!")

# Start training
for epoch in range(start_epoch, EPOCHS):

    # Unfreeze the last three feature blocks
    if epoch == FREEZE_EPOCHS:

        for param in model.features[-3:].parameters():
            param.requires_grad = True

        optimizer = torch.optim.AdamW(
            [
                {"params": model.features[-3:].parameters(), "lr": 1e-5},
                {"params": model.classifier.parameters(), "lr": 1e-4}
            ],
            weight_decay=1e-4
        )

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=2
        )

        print("Last three feature blocks unfrozen!")

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    # Training phase
    model.train()

    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    start_time = time.time()

    for batch_idx, (images, labels) in enumerate(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        # Show training progress
        if (batch_idx + 1) % 20 == 0 or batch_idx + 1 == len(train_loader):

            print(
                f"Training Batch [{batch_idx + 1}/{len(train_loader)}] "
                f"Loss: {loss.item():.4f} "
                f"Accuracy: {100 * correct / total:.2f}%"
            )

    train_loss = running_loss / len(train_loader)
    train_accuracy = 100 * correct / total

    # Validation phase
    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for batch_idx, (images, labels) in enumerate(val_loader):

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predicted.cpu().numpy())

            if (batch_idx + 1) % 20 == 0 or batch_idx + 1 == len(val_loader):
                print(
                    f"Validation Batch [{batch_idx + 1}/{len(val_loader)}]"
                )

    val_loss = val_loss / len(val_loader)
    val_accuracy = 100 * val_correct / val_total

    # Calculate evaluation metrics
    val_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro"
    )

    real_recall = recall_score(
        all_labels,
        all_predictions,
        pos_label=1,
        zero_division=0
    )

    # Update learning rate
    scheduler.step(val_f1)

    print("\nTraining Loss:", round(train_loss, 4))
    print("Training Accuracy:", round(train_accuracy, 2), "%")
    print("Validation Loss:", round(val_loss, 4))
    print("Validation Accuracy:", round(val_accuracy, 2), "%")
    print("Validation F1:", round(val_f1, 4))
    print("REAL Recall:", round(real_recall, 4))
    print("Epoch Time:", round((time.time() - start_time) / 60, 2), "minutes")

    # Save the best model
    if val_f1 > best_f1:

        best_f1 = val_f1
        best_val_loss = val_loss
        early_stop_count = 0

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "class_to_idx": train_ds.class_to_idx,
                "architecture": "efficientnet_b0",
                "epoch": epoch,
                "best_f1": best_f1,
                "best_val_loss": best_val_loss
            },
            BEST_MODEL_PATH
        )

        print("Best model saved!")

    else:
        early_stop_count += 1

    # Save latest checkpoint
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "epoch": epoch,
            "best_f1": best_f1,
            "best_val_loss": best_val_loss,
            "early_stop_count": early_stop_count
        },
        CHECKPOINT_PATH
    )

    print("Checkpoint saved!")

    # Stop if validation does not improve
    if early_stop_count >= PATIENCE:

        print("Early stopping!")
        break

print("\nTraining completed!")
print("Best Validation F1:", best_f1)


Starting new training!

Epoch 1/15
Training Batch [20/799] Loss: 0.6710 Accuracy: 45.47%
Training Batch [40/799] Loss: 0.7073 Accuracy: 48.05%
Training Batch [60/799] Loss: 0.6311 Accuracy: 51.72%
Training Batch [80/799] Loss: 0.6844 Accuracy: 55.39%
Training Batch [100/799] Loss: 0.6824 Accuracy: 57.00%
Training Batch [120/799] Loss: 0.6412 Accuracy: 57.45%
Training Batch [140/799] Loss: 0.7427 Accuracy: 58.21%
Training Batch [160/799] Loss: 0.6658 Accuracy: 59.10%
Training Batch [180/799] Loss: 0.6095 Accuracy: 60.09%
Training Batch [200/799] Loss: 0.7225 Accuracy: 60.53%
Training Batch [220/799] Loss: 0.7177 Accuracy: 61.05%
Training Batch [240/799] Loss: 0.6156 Accuracy: 61.54%
Training Batch [260/799] Loss: 0.7424 Accuracy: 61.85%


KeyboardInterrupt: 

In [17]:
import time

start = time.time()

for i, (images, labels) in enumerate(train_loader):
    if i == 20:
        break

print("Time for 20 batches:", round(time.time() - start, 2), "seconds")

Time for 20 batches: 152.68 seconds


In [19]:
!df -h /content

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G  443M  113G   1% /


In [21]:
SOURCE = "/content/drive/MyDrive/Deepfake_Project/celebdf_frames"
DESTINATION = "/content/celebdf_frames"

print("Copying dataset...")

shutil.copytree(
    SOURCE,
    DESTINATION,
    dirs_exist_ok=True
)

print("Dataset copied successfully!")

Copying dataset...
Dataset copied successfully!


In [24]:
LOCAL_TRAIN = "/content/celebdf_frames/train"
LOCAL_VAL = "/content/celebdf_frames/val"

# Load images from local storage
train_ds = train_ds.__class__(
    LOCAL_TRAIN,
    transform=train_ds.transform
)

val_ds = val_ds.__class__(
    LOCAL_VAL,
    transform=val_ds.transform
)

# Recreate DataLoaders
train_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Training samples:", len(train_ds))
print("Validation samples:", len(val_ds))
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

start = time.time()
for i, (images, labels) in enumerate(train_loader):
    if i == 20:
        break
print("Time for 20 batches:", round(time.time() - start, 2), "seconds")

!ls -lh /content/drive/MyDrive/Deepfake_Project/*.pth

Training samples: 25541
Validation samples: 4510
Training batches: 799
Validation batches: 141
Time for 20 batches: 1.54 seconds
-rw------- 1 root root 16M Nov 20 09:20 /content/drive/MyDrive/Deepfake_Project/best_model.pth
-rw------- 1 root root 47M Nov 20 09:20 /content/drive/MyDrive/Deepfake_Project/last_checkpoint.pth


In [27]:
from sklearn.metrics import f1_score, recall_score
# Original trained model
ORIGINAL_MODEL_PATH = "/content/drive/MyDrive/Deepfake_Project/best_model.pth"

# New fine-tuning files
BEST_MODEL_PATH = "/content/drive/MyDrive/Deepfake_Project/best_model_finetune.pth"
CHECKPOINT_PATH = "/content/drive/MyDrive/Deepfake_Project/last_checkpoint_finetune.pth"

# Load original checkpoint
original_checkpoint = torch.load(
    ORIGINAL_MODEL_PATH,
    map_location=device
)

# Create model without downloading ImageNet weights
model = models.efficientnet_b0(weights=None)

model.classifier[1] = nn.Linear(
    model.classifier[1].in_features, 2
)

# Load previously trained weights
if "model_state_dict" in original_checkpoint:
    model.load_state_dict(original_checkpoint["model_state_dict"])
else:
    model.load_state_dict(original_checkpoint)

model = model.to(device)

print("Original trained model loaded successfully!")

# Freeze all feature layers
for param in model.features.parameters():
    param.requires_grad = False

# Unfreeze the last three feature blocks
for param in model.features[-3:].parameters():
    param.requires_grad = True

# Keep classifier trainable
for param in model.classifier.parameters():
    param.requires_grad = True

# Define loss function
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Define optimizer
optimizer = torch.optim.AdamW(
    [
        {"params": model.features[-3:].parameters(), "lr": 1e-5},
        {"params": model.classifier.parameters(), "lr": 1e-5}
    ],
    weight_decay=1e-4
)

# Define scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

EPOCHS = 10
PATIENCE = 4

start_epoch = 0
best_f1 = 0.0
best_val_loss = float("inf")
early_stop_count = 0

print("Fine-tuning configuration ready!")

# Resume fine-tuning if a checkpoint exists
if os.path.exists(CHECKPOINT_PATH):

    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)

    model.load_state_dict(checkpoint["model_state_dict"])

    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    start_epoch = checkpoint["epoch"] + 1
    best_f1 = checkpoint["best_f1"]
    best_val_loss = checkpoint["best_val_loss"]
    early_stop_count = checkpoint["early_stop_count"]

    print("Fine-tuning checkpoint loaded!")
    print("Resuming from epoch:", start_epoch + 1)

else:
    print("Starting fine-tuning from original trained model!")

# Start training
for epoch in range(start_epoch, EPOCHS):

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    model.train()

    # Keep BatchNorm layers stable
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    start_time = time.time()

    # Training phase
    for batch_idx, (images, labels) in enumerate(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        # Show progress
        if (batch_idx + 1) % 20 == 0 or batch_idx + 1 == len(train_loader):

            print(
                f"Training [{batch_idx + 1}/{len(train_loader)}] "
                f"Loss: {loss.item():.4f} "
                f"Accuracy: {100 * correct / total:.2f}%"
            )

    train_loss = running_loss / len(train_loader)
    train_accuracy = 100 * correct / total

    # Validation phase
    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for batch_idx, (images, labels) in enumerate(val_loader):

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predicted.cpu().numpy())

            if (batch_idx + 1) % 20 == 0 or batch_idx + 1 == len(val_loader):
                print(
                    f"Validation [{batch_idx + 1}/{len(val_loader)}]"
                )

    val_loss /= len(val_loader)
    val_accuracy = 100 * val_correct / val_total

    # Calculate evaluation metrics
    val_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro"
    )

    real_recall = recall_score(
        all_labels,
        all_predictions,
        pos_label=1,
        zero_division=0
    )

    scheduler.step(val_f1)

    print("\nTraining Loss:", round(train_loss, 4))
    print("Training Accuracy:", round(train_accuracy, 2), "%")
    print("Validation Loss:", round(val_loss, 4))
    print("Validation Accuracy:", round(val_accuracy, 2), "%")
    print("Validation F1:", round(val_f1, 4))
    print("REAL Recall:", round(real_recall, 4))
    print("Epoch Time:", round((time.time() - start_time) / 60, 2), "minutes")

    # Save best model
    if val_f1 > best_f1:

        best_f1 = val_f1
        best_val_loss = val_loss
        early_stop_count = 0

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "class_to_idx": train_ds.class_to_idx,
                "architecture": "efficientnet_b0",
                "epoch": epoch,
                "best_f1": best_f1,
                "best_val_loss": best_val_loss
            },
            BEST_MODEL_PATH
        )

        print("Best model saved!")

    else:
        early_stop_count += 1

    # Save latest checkpoint
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "epoch": epoch,
            "best_f1": best_f1,
            "best_val_loss": best_val_loss,
            "early_stop_count": early_stop_count
        },
        CHECKPOINT_PATH
    )

    print("Checkpoint saved!")

    # Stop if validation does not improve
    if early_stop_count >= PATIENCE:

        print("Early stopping!")
        break

print("\nFine-tuning completed!")
print("Best Validation F1:", best_f1)

Original trained model loaded successfully!
Fine-tuning configuration ready!
Starting fine-tuning from original trained model!

Epoch 1/10
Training [20/799] Loss: 0.0785 Accuracy: 98.44%
Training [40/799] Loss: 0.0215 Accuracy: 98.59%
Training [60/799] Loss: 0.0357 Accuracy: 97.97%
Training [80/799] Loss: 0.0158 Accuracy: 97.58%
Training [100/799] Loss: 0.0064 Accuracy: 97.50%
Training [120/799] Loss: 0.1399 Accuracy: 97.34%
Training [140/799] Loss: 0.0243 Accuracy: 97.32%
Training [160/799] Loss: 0.0112 Accuracy: 97.44%
Training [180/799] Loss: 0.0324 Accuracy: 97.41%
Training [200/799] Loss: 0.0391 Accuracy: 97.50%
Training [220/799] Loss: 0.0019 Accuracy: 97.60%
Training [240/799] Loss: 0.1069 Accuracy: 97.51%
Training [260/799] Loss: 0.0564 Accuracy: 97.56%
Training [280/799] Loss: 0.0194 Accuracy: 97.62%
Training [300/799] Loss: 0.0389 Accuracy: 97.64%
Training [320/799] Loss: 0.0241 Accuracy: 97.66%
Training [340/799] Loss: 0.0130 Accuracy: 97.73%
Training [360/799] Loss: 0.0650 

In [28]:
# Load best fine-tuned model
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Best fine-tuned model loaded!")
print("Best validation F1:", checkpoint["best_f1"])

Best fine-tuned model loaded!
Best validation F1: 0.8445998992972669


## 9. Evaluate the best model on the official test set

In [33]:
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_path = "/content/celebdf_frames/test"

test_ds = datasets.ImageFolder(
    test_path,
    transform=val_transform
)

test_loader = DataLoader(
    test_ds,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Test samples:", len(test_ds))
print("Test classes:", test_ds.class_to_idx)


Test samples: 2590
Test classes: {'FAKE': 0, 'REAL': 1}


In [35]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
# Evaluate the fine-tuned model
model.eval()

all_labels = []
all_predictions = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        all_labels.extend(labels.numpy())
        all_predictions.extend(predicted.cpu().numpy())

# Display evaluation results
print("Classification Report:")
print(classification_report(
    all_labels,
    all_predictions,
    target_names=["FAKE", "REAL"]
))

print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_predictions))

print("Test Accuracy:", accuracy_score(all_labels, all_predictions))


Classification Report:
              precision    recall  f1-score   support

        FAKE       0.86      0.96      0.91      1700
        REAL       0.90      0.69      0.78       890

    accuracy                           0.87      2590
   macro avg       0.88      0.83      0.84      2590
weighted avg       0.87      0.87      0.86      2590

Confusion Matrix:
[[1632   68]
 [ 272  618]]
Test Accuracy: 0.8687258687258688


## 10. Upload a video and predict REAL or FAKE

In [36]:
from PIL import Image
from google.colab import files

# Load best model for inference
inference_checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)

inference_model = models.efficientnet_b0(weights=None)
inference_model.classifier[1] = nn.Linear(
    inference_model.classifier[1].in_features, 2
)
inference_model.load_state_dict(inference_checkpoint["model_state_dict"])
inference_model = inference_model.to(device)
inference_model.eval()

# Match the evaluation preprocessing used during training
inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ImageFolder class order is FAKE=0, REAL=1
class_to_idx = inference_checkpoint["class_to_idx"]
fake_index = class_to_idx["FAKE"]
real_index = class_to_idx["REAL"]

print("Model loaded successfully!")
print("Class mapping:", class_to_idx)

def predict_video(video_path, num_frames=20):
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        cap.release()
        raise ValueError("Could not open this video.")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        raise ValueError("Could not read frames from this video.")

    frame_indices = np.linspace(
        0, total_frames - 1,
        min(num_frames, total_frames),
        dtype=int
    )

    fake_probs = []

    for frame_index in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
        success, frame = cap.read()

        if not success:
            continue

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image = Image.fromarray(frame)
        image_tensor = inference_transform(image).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = inference_model(image_tensor)
            probabilities = torch.softmax(logits, dim=1)

        fake_probs.append(probabilities[0, fake_index].item())

    cap.release()

    if not fake_probs:
        raise ValueError("No readable frames were found.")

    avg_fake = float(np.mean(fake_probs))
    avg_real = 1.0 - avg_fake

    print(f"Frames analyzed: {len(fake_probs)}")
    print(f"Average FAKE score: {avg_fake:.2%}")
    print(f"Average REAL score: {avg_real:.2%}")

    if avg_fake >= 0.5:
        print("\nPrediction: FAKE")
    else:
        print("\nPrediction: REAL")

    return avg_fake

uploaded = files.upload()

if uploaded:
    video_path = next(iter(uploaded.keys()))
    fake_score = predict_video(video_path)


Model loaded successfully!
Class mapping: {'FAKE': 0, 'REAL': 1}


Saving Haaland Watermelon Magic Trick! 🍉🌭 #football #parody #haaland.mp4 to Haaland Watermelon Magic Trick! 🍉🌭 #football #parody #haaland.mp4
Frames analyzed: 20
Average FAKE score: 92.82%
Average REAL score: 7.18%

Prediction: FAKE
